In [ ]:
import yfinance as yf
import pandas as pd
import os
import requests  # <-- LIBRERÍA NUEVA

# --- 1. Obtener la lista de Tickers del S&P 500 ---
print("Obteniendo la lista de tickers del S&P 500 desde Wikipedia...")
try:
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    
    # 1. Pretendemos ser un navegador web (User-Agent)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36'
    }
    
    # 2. Usamos 'requests' para obtener el HTML de forma segura
    response = requests.get(url, headers=headers)
    
    # 3. Leemos el HTML desde el texto de la respuesta
    tabla = pd.read_html(response.text)

    df_tickers = tabla[0]
    tickers = df_tickers['Symbol'].tolist()
    tickers = [s.replace('.', '-') for s in tickers]
    
    print(f"Lista obtenida. {len(tickers)} tickers encontrados.")

except Exception as e:
    print(f"Error al obtener la lista de tickers: {e}")
    exit() # Detener si no podemos obtener los tickers

# --- 2. Definir Fechas (Basado en la Guía) ---
start_date = '2018-01-02'
end_date = '2019-10-12'

# --- 3. Descargar los Datos de Precios ---
print(f"Descargando datos de precios para {len(tickers)} tickers...")

try:
    full_data = yf.download(tickers, start=start_date, end=end_date)

    if full_data.empty:
        print("\n--- ERROR: La descarga falló. yfinance devolvió una tabla vacía ---")
    
    else:
        print("\nDescarga completa. Extrayendo precios de Cierre (Ajustados)...")
        
        data = full_data.get('Close') 
        
        if data is None:
            print("Error: No se encontró la columna 'Close'.")
        else:
            output_dir = 'inputs'
            file_path = os.path.join(output_dir, 'sp500_closes.csv')

            if not os.path.exists(output_dir):
                os.makedirs(output_dir)
                print(f"Carpeta '{output_dir}' creada.")

            data.to_csv(file_path)
            print(f"\n¡Éxito! Datos guardados en: {file_path}")
            
            print("\nVerificación (primeras 5 filas):")
            print(data.head())

except Exception as e:
    print(f"\nOcurrió un error inesperado durante la descarga: {e}")